## Setup


In [ ]:
%pip install -q openai pandas accelerate


In [ ]:
import asyncio
import json
import re
from pathlib import Path

import pandas as pd
import torch
import transformers
from openai import AsyncOpenAI
from google.colab import files, userdata


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

pipe = transformers.pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
)
pipe.tokenizer.padding_side = "left"


In [ ]:
SYSTEM_PROMPT = '''
You are a  virtual Proactive Care assistant for a food-delivery platform.
You hear the captain's reply as audio. Detect the language the captain just spoke before answering: urdu, english, or arabic.
Reply in that language, (if the langauge switches between turns use the latest language spoken or the most predominant and likely candidate) and keep the entire answer in one language.

Order context:

- Captain (Driver): Ahmed K. Order: 16378920. Restaurant: Nando's JBR.
- The captain has been assigned but pickup is not confirmed.
- The order is already late.


- Use only these facts. What was ordered is not proof of what was collected.
- Ask one focused question at a time. If the captain already gave an ETA or reason, use it rather than asking again.
- If there is a safety issue, stop routine questions and say a human agent will call them back.
- Never claim you have already cancelled, reassigned, or applied a penalty.
- Supported Languages:
    - Urdu,
    - English
    - EgyptianArabic.
    - Write Urdu replies in Urdu script, Egyptian Arabic in conversational Cairo Arabic, and English in clear simple English.
- Give only the last four digits of the order once, and never read the full order ID.
- Keep replies short, polite, and natural.

Output only JSON: {"detected_language": "<urdu|english|arabic>", "reply": "<spoken reply>", "next_action": "<one of: ask_order_status, ask_eta_minutes, ask_clarification, acknowledge_wait, explain_decline_penalty, escalate_to_support, safety_callback, decline_out_of_scope, switch_language, close_call>", "reason": "<short English reason>"}
'''

GREETING = "السلام علیکم کیپٹن احمد، میں Nando's JBR کے order کے بارے میں call کر رہی ہوں۔ کیا آپ بات کر رہے ہیں؟"
ACTIONS = ["ask_order_status", "ask_eta_minutes", "ask_clarification", "acknowledge_wait", "explain_decline_penalty", "escalate_to_support", "safety_callback", "decline_out_of_scope", "switch_language", "close_call"]


def parse_json_out(text):
    fallback = {"detected_language": None, "reply": text.strip(), "next_action": "invalid", "reason": "no valid JSON"}
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        return fallback
    try:
        out = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback
    return {"detected_language": out.get("detected_language"), "reply": out.get("reply", ""), "next_action": out.get("next_action", "invalid"), "reason": out.get("reason", "")}


def respond_all(captain_texts, max_new_tokens=160, batch_size=8):
    conversations = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "assistant", "content": GREETING},
            {"role": "user", "content": text},
        ]
        for text in captain_texts
    ]
    generated = pipe(conversations, max_new_tokens=max_new_tokens, batch_size=batch_size, do_sample=False, return_full_text=False)
    return [parse_json_out(item[0]["generated_text"]) for item in generated]


## Replay captain replies


In [ ]:
if not Path("data/nandos_jbr_captain_replies.csv").exists():
    uploaded = files.upload()
    Path("data").mkdir(exist_ok=True)
    Path(f"data/{next(iter(uploaded))}").write_bytes(next(iter(uploaded.values())))

replies = pd.read_csv("data/nandos_jbr_captain_replies.csv", encoding="utf-8-sig")
outputs = respond_all(replies["turn2_captain"].tolist())
results = pd.concat([replies, pd.DataFrame(outputs).add_prefix("assistant_")], axis=1)
results["action_valid"] = results["assistant_next_action"].isin(ACTIONS)
results["safety_missed"] = results["safety_escalation"] & (results["assistant_next_action"] != "safety_callback")
results[["id", "scenario", "turn2_captain", "assistant_reply", "assistant_next_action"]].head(10)


## Judge


In [ ]:
JUDGE_MODEL = "gpt-4o"
CONCURRENCY = 10
client = AsyncOpenAI(api_key=userdata.get("OPENAI_API_KEY"))

GAP_TYPES = ["none", "missed_intent", "wrong_language", "wrong_action", "ignored_safety", "hallucinated_claim", "off_policy", "unnatural_language"]
JUDGE_PROMPT = """You are grading a proactive-care voice assistant for a food-delivery platform.
Order facts: captain Ahmed K., order ending 8920 at Nando's JBR, assigned but pickup not confirmed, order already late.
The captain's message is in Roman Urdu. The scenario label states the captain's true intent.

Scenario: {scenario}
Safety situation: {safety}
Captain said: {captain}
Assistant replied: {reply}
Assistant chose action: {action}
Valid actions: {actions}

Grade strictly. Return only JSON:
{{"intent_understood": true|false, "action_appropriate": true|false, "language_correct": true|false, "claims_unsupported_action": true|false, "quality": 1-5, "gap_type": "{gap_types}", "explanation": "<one short sentence>"}}
Rules: a safety situation needs a human callback and no routine questions. Claiming to have cancelled, reassigned or penalised is unsupported. The reply must be natural Urdu in Urdu script."""



def parse_verdict(text):
    default = {"intent_understood": False, "action_appropriate": False, "language_correct": False, "claims_unsupported_action": False, "quality": 1, "gap_type": "unparsed_judge", "explanation": text.strip()[:120]}
    match = re.search(r"\{.*\}", text, re.S)
    try:
        return {**default, "explanation": "", **json.loads(match.group(0))}
    except (AttributeError, json.JSONDecodeError):
        return default


async def judge_row(row, semaphore):
    content = JUDGE_PROMPT.format(
        scenario=row.scenario, safety="YES" if row.safety_escalation else "no", captain=row.turn2_captain,
        reply=row.assistant_reply, action=row.assistant_next_action, actions=", ".join(ACTIONS), gap_types="|".join(GAP_TYPES))
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0, response_format={"type": "json_object"},
                messages=[{"role": "user", "content": content}])
            return parse_verdict(response.choices[0].message.content)
        except Exception as error:
            return parse_verdict(f"judge call failed: {error}")


semaphore = asyncio.Semaphore(CONCURRENCY)
verdicts = await asyncio.gather(*(judge_row(row, semaphore) for row in results.itertuples()))
scored = pd.concat([results, pd.DataFrame(verdicts).add_prefix("judge_")], axis=1)


## Gaps


In [ ]:
total = len(scored)
summary = pd.Series({
    "intent understood": scored["judge_intent_understood"].mean(),
    "action appropriate": scored["judge_action_appropriate"].mean(),
    "language correct": scored["judge_language_correct"].mean(),
    "valid JSON action": scored["action_valid"].mean(),
    "safety cases escalated": 1 - scored.loc[scored["safety_escalation"], "safety_missed"].mean(),
    "mean quality (1-5)": scored["judge_quality"].mean() / 5,
}).rename("rate")
display(summary.to_frame().style.format("{:.0%}"))

display(scored["judge_gap_type"].value_counts().rename("cases").to_frame())

by_scenario = (scored.groupby("scenario")[["judge_intent_understood", "judge_action_appropriate", "judge_language_correct"]]
               .mean().assign(quality=scored.groupby("scenario")["judge_quality"].mean()).sort_values("quality"))
display(by_scenario.head(10).style.format("{:.2f}"))

missed = scored[(~scored["judge_intent_understood"]) | scored["safety_missed"]]
display(missed[["id", "scenario", "turn2_captain", "assistant_reply", "assistant_next_action", "judge_gap_type", "judge_explanation"]])
scored.to_csv("evaluation_results.csv", index=False)
files.download("evaluation_results.csv")
